# ASG Airlines: End-to-End Data Engineering Pipeline

**Objective:** Ingest, clean, standardize and model ASG Airlines' operational data
(flights, bookings, passengers, payments) into an analytics-ready dataset for
Power BI reporting.

**Architecture:** `Excel source -> Python ETL (this notebook, bronze/silver/gold layers)
-> Local MySQL Server -> Power BI Desktop`

Python (pandas) handles ingestion, cleaning, and modelling. A local MySQL Server
holds the data in two databases: `asg_airlines` for reporting and
`asg_airlines_restricted` for PII, with access control enforced through MySQL
users and GRANT/REVOKE. This keeps the full pipeline runnable on a single
machine with no cloud account or setup step required, and MySQL's user and
privilege system gives a real, testable access boundary between the reporting
data and the restricted PII, rather than a boundary enforced only by
convention.

A scripted Python pipeline is used for the transform layer rather than a
platform like Azure Data Factory or Databricks. For a dataset this size (about
4,000 rows across 4 files), this is faster to build and test, and keeps every
transformation decision inspectable as code.

## 1. Ingestion (bronze layer)

Load all 4 source sheets as provided, with no transformation yet, so the raw
input stays reproducible from this notebook.

In [1]:
import pandas as pd
import numpy as np
import hashlib
import json

SRC = "UseCase_-_Airlines.xlsx"   # place the provided Excel file next to this notebook
SALT = "asg-airlines-2026"  # use a secrets vault in production

flights = pd.read_excel(SRC, sheet_name="flights")
bookings = pd.read_excel(SRC, sheet_name="bookings")
passengers = pd.read_excel(SRC, sheet_name="passengers")
payments = pd.read_excel(SRC, sheet_name="payments")

dq_log = {}
dq_log["raw_row_counts"] = {
    "flights": len(flights), "bookings": len(bookings),
    "passengers": len(passengers), "payments": len(payments),
}
dq_log["raw_row_counts"]

{'flights': 1020, 'bookings': 1000, 'passengers': 1039, 'payments': 1000}

## 2. Data quality discovery -> cleaning decisions

The raw data was profiled before writing any cleaning logic. This surfaced issues
beyond what the brief mentions, each handled with an explicit rule rather than a
blanket "drop everything unclean" approach:

| Issue found | Rows affected | Decision |
|---|---|---|
| Exact duplicate flight rows | 15 | Dropped, keep first occurrence |
| **Same `flight_id` reused for two genuinely different flights** (different times) | 2 IDs / 4 rows | Kept both, given a synthetic surrogate key (`flight_sk`); not deduped away, since that would silently discard a real flight |
| `airline` missing / literally `"UNKNOWN"` | 69 | Imputed deterministically from the 2-letter `flight_id` prefix (SJ to SpiceJet, AI to Air India, UK to Vistara, 6F to IndiGo), evidence-based, not guessed |
| Arrival timestamp earlier than departure with **no** date rollover (logically impossible, not a normal overnight case) | 1 | Flagged `corrupted_timestamp`, duration left null, excluded from duration KPIs but counted in the anomaly log |
| Normal overnight flights (arrival date = departure date + 1) | 124 | Duration computed directly from the timestamp difference, no special-casing needed since the date already rolls over correctly |
| Fully blank rows in `bookings` | 12 (auto-dropped by pandas on read) | Verified as junk rows, not real bookings |
| `status` missing (`None`) vs. literal `"INVALID"` | 45 vs. 30 | Treated as two different things: `None` becomes `UNKNOWN`; `"INVALID"` kept as its own status because it may be a real business state, not just bad data entry |
| Duplicate `passenger_id` with **conflicting** details (whitespace, different email/phone/Aadhaar/DOB) | 39 | Kept the most complete record per ID (fewest nulls, ties broken by last occurrence); conflict count logged as a data-quality metric |
| Missing `payment.amount` | 48 | Left null and excluded from revenue KPIs rather than imputed, since inventing a number would bias revenue reporting |
| `payment.amount` literally the string `"INVALID"` | 30 | Coerced to null and tracked as a separate metric from missing. Matches the same `"INVALID"` placeholder pattern found in `bookings.status`, confirming it's a deliberate corruption pattern in the source system, not a one-off |

In [2]:
# ---- Flights ----
before = len(flights)
flights = flights.drop_duplicates(subset=["flight_id", "departure_time", "arrival_time"], keep="first")
dq_log["flights_exact_duplicates_removed"] = before - len(flights)

# Reused/corrupted identifiers: same flight_id, different times -> 2 distinct flights sharing 1 id
id_conflict_mask = flights.duplicated(subset=["flight_id"], keep=False)
dq_log["flights_reused_id_conflicts"] = int(id_conflict_mask.sum())
flights["flight_sk"] = flights["flight_id"]
flights.loc[id_conflict_mask, "flight_sk"] = (
    flights.loc[id_conflict_mask, "flight_id"] + "_" +
    flights.loc[id_conflict_mask].groupby("flight_id").cumcount().add(1).astype(str)
)
flights["ambiguous_id"] = id_conflict_mask

# Airline imputation from flight_id prefix
prefix_map = {"SJ": "SpiceJet", "AI": "Air India", "UK": "Vistara", "6F": "IndiGo"}
missing_airline_mask = flights["airline"].isna() | (flights["airline"] == "UNKNOWN")
dq_log["flights_airline_imputed"] = int(missing_airline_mask.sum())
flights.loc[missing_airline_mask, "airline"] = flights.loc[missing_airline_mask, "flight_id"].str[:2].map(prefix_map)

flights["departure_time"] = pd.to_datetime(flights["departure_time"])
flights["arrival_time"] = pd.to_datetime(flights["arrival_time"])
flights["is_overnight"] = flights["arrival_time"].dt.date > flights["departure_time"].dt.date

raw_diff_min = (flights["arrival_time"] - flights["departure_time"]).dt.total_seconds() / 60
flights["data_quality_flag"] = np.where(raw_diff_min <= 0, "corrupted_timestamp", "clean")
flights["duration_minutes"] = np.where(raw_diff_min > 0, raw_diff_min, np.nan)
dq_log["flights_corrupted_timestamps"] = int((flights["data_quality_flag"] == "corrupted_timestamp").sum())

flights[["flight_id", "flight_sk", "airline", "is_overnight", "data_quality_flag", "duration_minutes"]].head()

,flight_id,flight_sk,airline,is_overnight,data_quality_flag,duration_minutes
0,SJ010,SJ010,SpiceJet,True,clean,174.0
1,AI155,AI155,Air India,True,clean,108.0
2,UK094,UK094,Vistara,True,clean,105.0
3,AI245,AI245,Air India,True,clean,156.0
4,AI192,AI192,Air India,True,clean,299.0


### On "Delays / Anomalies"

The source data has no scheduled-time field, only actual `departure_time` and
`arrival_time`. A true delay (actual vs. scheduled) can't be computed from this
dataset, so none is reported.

As a check, flight duration was tested for statistical outliers per route (an
IQR test). It found zero outliers: duration is spread uniformly from 30 to 300
minutes on every route, with no natural clustering, which suggests this field
was randomly generated for the exercise rather than modeled on real scheduling
patterns. The detector stays in the pipeline since it would be valid on
real-world data, but this result is reported as found rather than overstated.

Given that, "Anomalies" for this dataset means the data-quality anomalies
actually present: corrupted timestamps, reused flight identifiers, and bookings
that reference an ambiguous flight ID.

In [3]:
flights["route"] = flights["source"] + "-" + flights["destination"]

def flag_route_outliers(g):
    q1, q3 = g["duration_minutes"].quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return (g["duration_minutes"] < lo) | (g["duration_minutes"] > hi)

flights["is_duration_anomaly"] = False
clean_mask = flights["data_quality_flag"] == "clean"
flights.loc[clean_mask, "is_duration_anomaly"] = (
    flights[clean_mask].groupby("route", group_keys=False).apply(flag_route_outliers)
)
dq_log["flights_duration_anomalies"] = int(flights["is_duration_anomaly"].sum())
print("Route-level duration outliers found:", dq_log["flights_duration_anomalies"])

Route-level duration outliers found: 0


In [4]:
# ---- Bookings ----
before = len(bookings)
bookings = bookings.dropna(how="all")
dq_log["bookings_blank_rows_removed"] = before - len(bookings)

bookings["status"] = bookings["status"].fillna("UNKNOWN")  # INVALID kept as its own state
dq_log["bookings_status_imputed_unknown"] = int((bookings["status"] == "UNKNOWN").sum())

ambiguous_ids = set(flights.loc[flights["ambiguous_id"], "flight_id"])
bookings["ambiguous_flight_ref"] = bookings["flight_id"].isin(ambiguous_ids)
dq_log["bookings_with_ambiguous_flight_ref"] = int(bookings["ambiguous_flight_ref"].sum())

bookings["status"].value_counts()

status
CONFIRMED    320
CANCELLED    314
PENDING      291
UNKNOWN       45
INVALID       30
Name: count, dtype: int64

## 3. PII masking (passengers)

`passengers` contains genuine PII: Aadhaar ID, passport number (in `bookings`),
email, phone, date of birth, and emergency contact details. Two techniques are
used, matched to how each field is used downstream:

- **One-way SHA-256 hash** for Aadhaar ID / passport number: irreversible, but
  the same real value always hashes to the same output, so duplicate-person
  checks still work without storing the real number in the analytics layer.
- **Partial masking** for email/phone: keeps them recognisable for support
  cases (`v***@gmail.com`) without exposing the full value.
- Exact date of birth is dropped from the analytics table and replaced with an
  `age_band` bucket, since exact DOB plus name is a re-identification risk even
  after other fields are masked.
- Full name, exact DOB, and emergency contact details are kept only in a
  separate, access-restricted table (`secure_booking_pii`), never in the
  tables Power BI or general analysts touch. See the Privacy and access
  control section for how this is enforced at the database level.

In [5]:
def hash_pii(value, salt=SALT):
    if pd.isna(value):
        return None
    return hashlib.sha256(f"{salt}{value}".encode()).hexdigest()[:16]

def mask_email(email):
    if pd.isna(email):
        return None
    try:
        user, domain = email.split("@")
        return user[0] + "***@" + domain
    except Exception:
        return "***"

def mask_phone(phone):
    if pd.isna(phone):
        return None
    s = str(phone)
    return s[:3] + "-XXXXX" + s[-2:]

# ---- Passengers: de-duplicate first, then mask ----
passengers["first_name"] = passengers["first_name"].str.strip()
passengers["last_name"] = passengers["last_name"].str.strip()

before = len(passengers)
passengers["_completeness"] = passengers.notna().sum(axis=1)
passengers = passengers.sort_values(["passenger_id", "_completeness"]).drop_duplicates(subset="passenger_id", keep="last")
dq_log["passengers_duplicate_ids_resolved"] = before - len(passengers)
passengers = passengers.drop(columns=["_completeness"])

dim_passenger = passengers.copy()
dim_passenger["aadhaar_hash"] = dim_passenger["aadhaar_id"].apply(hash_pii)
dim_passenger["email_masked"] = dim_passenger["email"].apply(mask_email)
dim_passenger["phone_masked"] = dim_passenger["phone"].apply(mask_phone)
dim_passenger["age_band"] = pd.cut(dim_passenger["age"], bins=[0, 17, 25, 40, 60, 120],
                                     labels=["<18", "18-25", "26-40", "41-60", "60+"])
dim_passenger = dim_passenger.drop(columns=["aadhaar_id", "email", "phone", "date_of_birth", "first_name", "last_name"])
dim_passenger.head(3)

,passenger_id,age,gender,aadhaar_hash,email_masked,phone_masked,age_band
0,P1000,52,F,dbc976463c58cebc,v***@gmail.com,+91-XXXXX90,41-60
1,P1001,15,M,770d0e3801d16861,k***@hotmail.com,+91-XXXXX97,<18
2,P1002,72,M,d94d5c74b1b87dfe,m***@outlook.com,+91-XXXXX92,60+


In [6]:
# ---- Payments ----
# 'amount' mixes real numbers with a literal "INVALID" placeholder (see table above)
dq_log["payments_missing_amount"] = int(payments["amount"].isna().sum())
dq_log["payments_invalid_amount_string"] = int((payments["amount"] == "INVALID").sum())
payments["amount"] = pd.to_numeric(payments["amount"], errors="coerce")
payments["payment_method"] = payments["payment_method"].str.upper().str.strip()
payments["payment_method"].value_counts()

payment_method
UPI           358
CARD          329
NETBANKING    313
Name: count, dtype: int64

## 4. Gold layer: star schema for reporting

A small star schema: one fact table per business event (flights operated,
bookings made, payments received), a couple of dimension tables for
enrichment, and one access-restricted table kept out of the reporting layer
entirely.

In [7]:
airport_names = {
    "CCU": "Kolkata", "BOM": "Mumbai", "MAA": "Chennai",
    "DEL": "Delhi", "BLR": "Bengaluru", "HYD": "Hyderabad",
}
dim_airport = pd.DataFrame({"airport_code": list(airport_names), "city": list(airport_names.values())})

fact_flights = flights[[
    "flight_sk", "flight_id", "ambiguous_id", "airline", "source", "destination", "route",
    "departure_time", "arrival_time", "duration_minutes", "is_overnight",
    "data_quality_flag", "is_duration_anomaly"
]].copy()

fact_bookings = bookings.rename(columns={"flight_id": "flight_id_ref"})[[
    "booking_id", "passenger_id", "flight_id_ref", "booking_date", "status",
    "seat_number", "ambiguous_flight_ref"
]].copy()

# Restricted table: real PII, not exposed to the general reporting layer (see docs)
secure_booking_pii = bookings[["booking_id", "passport_number", "emergency_contact_name", "emergency_contact_phone"]].copy()

fact_payments = payments.copy()

for name, df in [("fact_flights", fact_flights), ("fact_bookings", fact_bookings),
                  ("fact_payments", fact_payments), ("dim_passenger", dim_passenger),
                  ("dim_airport", dim_airport), ("secure_booking_pii", secure_booking_pii)]:
    print(f"{name}: {len(df)} rows, {len(df.columns)} cols")

fact_flights: 1005 rows, 13 cols
fact_bookings: 1000 rows, 7 cols
fact_payments: 1000 rows, 4 cols
dim_passenger: 1000 rows, 7 cols
dim_airport: 6 rows, 2 cols
secure_booking_pii: 1000 rows, 4 cols


## 5. Business KPIs

Required KPIs plus a few additional ones that fell naturally out of having 4 linked
tables instead of just 1 (revenue and booking-funnel metrics need the
flights + bookings + payments join).

In [8]:
# Average flight duration (overall, by airline, by route), corrupted rows excluded
clean_flights = fact_flights[fact_flights["data_quality_flag"] == "clean"]

print("Avg duration overall (min):", round(clean_flights["duration_minutes"].mean(), 1))
print("\nAvg duration by airline:")
print(clean_flights.groupby("airline")["duration_minutes"].mean().round(1))
print("\nTop 5 busiest routes:")
print(clean_flights["route"].value_counts().head(5))
print("\nFlights by airline (distribution):")
print(fact_flights["airline"].value_counts())

Avg duration overall (min): 164.5

Avg duration by airline:
airline
Air India    165.4
IndiGo       164.9
SpiceJet     163.2
Vistara      164.3
Name: duration_minutes, dtype: float64

Top 5 busiest routes:
route
BOM-CCU    90
CCU-DEL    72
MAA-BLR    65
BLR-BOM    60
HYD-MAA    57
Name: count, dtype: int64

Flights by airline (distribution):
airline
IndiGo       273
Air India    255
SpiceJet     247
Vistara      230
Name: count, dtype: int64


In [9]:
# Anomaly summary
anomaly_summary = pd.Series({
    "corrupted_timestamp_flights": (fact_flights["data_quality_flag"] == "corrupted_timestamp").sum(),
    "reused_flight_id_conflicts": fact_flights["ambiguous_id"].sum(),
    "route_duration_outliers": fact_flights["is_duration_anomaly"].sum(),
    "bookings_with_ambiguous_flight_ref": fact_bookings["ambiguous_flight_ref"].sum(),
})
anomaly_summary

corrupted_timestamp_flights           1
reused_flight_id_conflicts            2
route_duration_outliers               0
bookings_with_ambiguous_flight_ref    2
dtype: int64

In [10]:
# Revenue & booking-funnel KPIs (join bookings -> payments, and bookings -> flights)
bp = fact_bookings.merge(fact_payments, on="booking_id", how="left")
bpf = bp.merge(fact_flights, left_on="flight_id_ref", right_on="flight_id", how="left")

print("Booking status funnel:")
print(fact_bookings["status"].value_counts(normalize=True).round(3) * 100)

print("\nRevenue by airline (CONFIRMED bookings only, INR):")
print(bpf[bpf["status"] == "CONFIRMED"].groupby("airline")["amount"].sum().round(0))

print("\nPayment method mix:")
print(fact_payments["payment_method"].value_counts(normalize=True).round(3) * 100)

Booking status funnel:
status
CONFIRMED    32.0
CANCELLED    31.4
PENDING      29.1
UNKNOWN       4.5
INVALID       3.0
Name: proportion, dtype: float64

Revenue by airline (CONFIRMED bookings only, INR):
airline
Air India    509566.0
IndiGo       493193.0
SpiceJet     697587.0
Vistara      771056.0
Name: amount, dtype: float64

Payment method mix:
payment_method
UPI           35.8
CARD          32.9
NETBANKING    31.3
Name: proportion, dtype: float64


## 6. Write outputs

These CSVs are what get loaded into the local MySQL server next (see
load_to_mysql.py). Keeping this as separate files, rather than one flat
export, mirrors how the star schema looks as real tables.

In [11]:
fact_flights.to_csv("fact_flights.csv", index=False)
fact_bookings.to_csv("fact_bookings.csv", index=False)
fact_payments.to_csv("fact_payments.csv", index=False)
dim_passenger.to_csv("dim_passenger.csv", index=False)
dim_airport.to_csv("dim_airport.csv", index=False)
secure_booking_pii.to_csv("secure_booking_pii.csv", index=False)

with open("data_quality_log.json", "w") as f:
    json.dump(dq_log, f, indent=2, default=str)

dq_log

{'raw_row_counts': {'flights': 1020,
  'bookings': 1000,
  'passengers': 1039,
  'payments': 1000},
 'flights_exact_duplicates_removed': 15,
 'flights_reused_id_conflicts': 2,
 'flights_airline_imputed': 69,
 'flights_corrupted_timestamps': 1,
 'flights_duration_anomalies': 0,
 'bookings_blank_rows_removed': 0,
 'bookings_status_imputed_unknown': 45,
 'bookings_with_ambiguous_flight_ref': 2,
 'passengers_duplicate_ids_resolved': 39,
 'payments_missing_amount': 48,
 'payments_invalid_amount_string': 30}

## 7. Privacy and access control

- `secure_booking_pii` (passport numbers, emergency contacts) and the
  pre-masking passenger fields are loaded into `asg_airlines_restricted`, a
  physically separate MySQL database from the reporting database
  (`asg_airlines`), not just a separate schema or table. See
  load_to_mysql.py and mysql_governance.sql.
- Aadhaar ID is never stored in plaintext past this notebook's in-memory
  step; only the SHA-256 hash is persisted.
- A dedicated `reporting_reader` MySQL user is granted SELECT on
  `asg_airlines` only; no GRANT of any kind is issued on
  `asg_airlines_restricted`, so the access boundary is enforced by MySQL's
  own privilege system rather than by convention. Tested directly:
  connecting as `reporting_reader` against `asg_airlines_restricted` returns
  "Access denied for user" (MySQL error 1044), while the same user queries
  `asg_airlines` normally.
- Net effect: Power BI, connected as `reporting_reader`, can only reach
  `dim_passenger` (already masked) and the fact tables, never
  `secure_booking_pii` or a plaintext Aadhaar/passport number.